# Documentación del Proceso ETL: Consolidación Histórica y Enriquecimiento Espacial del Valor de Referencia

## 1. Fuentes de Datos e Ingesta

El proyecto se alimenta de tres fuentes oficiales abiertas provistas por el Distrito Capital a través del portal de Datos Abiertos Bogotá e IDECA:

Valor de Referencia por Metro Cuadrado de Terreno: Conjunto de datos históricos en formato comprimido (.zip) que agrupa los polígonos de manzanas catastrales y sus respectivos valores económicos de referencia calculados año tras año (2012–2026).

Fuente: https://datosabiertos.bogota.gov.co/dataset/valor-de-referencia-por-metro-cuadrado-de-terreno

Sector Catastral: Capa geográfica oficial (.zip / Shapefile) que define la división político-administrativa a nivel de barrios o sectores catastrales, utilizada para contextualizar espacialmente cada manzana.

Fuente: https://datosabiertos.bogota.gov.co/dataset/sector-catastral

Localidad (loca.gpkg): Capa geoespacial en formato GeoPackage que delimita las localidades del Distrito Capital, permitiendo establecer la jerarquía territorial superior.

Fuente: https://datosabiertos.bogota.gov.co/dataset/localidad-bogota-d-c

## 2. Metodología y Procesamiento (ETL Pipeline)

El pipeline automatizado implementa un proceso de Extracción, Transformación y Carga (ETL) espacial estructurado en los siguientes pasos:

Ingestión y Estandarización: Se descomprimen de forma masiva los archivos históricos anuales, extrayendo las geometrías de las manzanas y normalizando los nombres de columnas heterogéneas (mancodigo, v_ref, shape_area, shape_len, ano) para unificarlas en un único marco de datos temporal.

Enriquecimiento Espacial por Intersección (sjoin): Utilizando la librería geopandas, se reproyectan las capas bajo un mismo sistema de referencia de coordenadas (CRS) y se ejecutan cruces espaciales secuenciales basados en el predicado intersects. Esto permite asignar a cada manzana histórica su respectivo barrio (desde el Sector Catastral) y localidad (desde el archivo GeoPackage).

Deduplicación e Integridad Estructural: Para corregir la multiplicación de registros derivada de manzanas que intersectan bordes administrativos, se implementa una limpieza basada en el identificador único original (id_original), garantizando una única asignación espacial por manzana.

Depuración de Datos Incompletos: Se aplica un filtro de exclusión (dropna) para remover registros que carezcan de variables económicas o administrativas esenciales (barrio, localidad, v_ref, shape_area, shape_len).

## 3. Productos Finales Generados

Como resultado del procesamiento, se obtienen dos datasets estructurados:

valor_referencia_historico_tabular.csv: Archivo tabular optimizado para análisis estadísticos y econométricos. Contiene el histórico consolidado a nivel de manzana, con sus valores de referencia, dimensiones físicas y jerarquía administrativa (barrio y localidad).

valor_referencia_historico_espacial.gpkg: Archivo GeoPackage que conserva la geometría vectorial completa, permitiendo visualizaciones geográficas y análisis de autocorrelación espacial en software SIG (QGIS, ArcGIS) o bibliotecas de Python.

In [ ]:
import os
import zipfile
import glob
import re
import pandas as pd
import geopandas as gpd
from osgeo import gdal

gdal.SetConfigOption("SHAPE_RESTORE_SHX", "YES")

os.makedirs('data_historica', exist_ok=True)
for f in glob.glob('*.zip'):
    if 'sector' not in f.lower():
        with zipfile.ZipFile(f, 'r') as z:
            z.extractall(os.path.join('data_historica', os.path.splitext(f)[0]))

os.makedirs('temp_sector', exist_ok=True)
for f in glob.glob('*sector*.zip'):
    with zipfile.ZipFile(f, 'r') as z:
        z.extractall('temp_sector')

lista_gdfs = []
for shp in glob.glob('data_historica/**/*.shp', recursive=True):
    gdf = gpd.read_file(shp)
    match = re.search(r'20\d{2}', shp)
    ano = int(match.group(0)) if match else None
    cols = {c.lower(): c for c in gdf.columns}
    datos = {
        'mancodigo': gdf[cols.get('mancodigo')] if cols.get('mancodigo') else None,
        'v_ref': gdf[cols.get('v_ref') or cols.get('valor_refe') or cols.get('valref')] if (cols.get('v_ref') or cols.get('valor_refe') or cols.get('valref')) else None,
        'shape_area': gdf[cols.get('shape_area')] if cols.get('shape_area') else None,
        'shape_len': gdf[cols.get('shape_leng') or cols.get('shape_len')] if (cols.get('shape_leng') or cols.get('shape_len')) else None,
        'ano': ano,
        'geometry': gdf['geometry']
    }
    lista_gdfs.append(gpd.GeoDataFrame(datos, crs=gdf.crs))

gdf_maestro = pd.concat(lista_gdfs, ignore_index=True)
gdf_maestro = gpd.GeoDataFrame(gdf_maestro, geometry='geometry')
gdf_maestro = gdf_maestro.reset_index().rename(columns={'index': 'id_original'})
print(f"1. Registros en gdf_maestro inicial: {len(gdf_maestro)}")

archivo_sector = glob.glob('temp_sector/**/*.shp', recursive=True)[0]
gdf_sector = gpd.read_file(archivo_sector)
if gdf_maestro.crs != gdf_sector.crs:
    gdf_sector = gdf_sector.to_crs(gdf_maestro.crs)

gdf_sector_sub = gdf_sector[['SCANOMBRE', 'geometry']].rename(columns={'SCANOMBRE': 'barrio'})
gdf_enriquecido = gpd.sjoin(gdf_maestro, gdf_sector_sub, how='left', predicate='intersects')
if 'index_right' in gdf_enriquecido.columns:
    gdf_enriquecido = gdf_enriquecido.drop(columns=['index_right'])
print(f"2. Registros tras sjoin con barrio: {len(gdf_enriquecido)}")

gdf_localidad = gpd.read_file('loca.gpkg')
if gdf_enriquecido.crs != gdf_localidad.crs:
    gdf_localidad = gdf_localidad.to_crs(gdf_enriquecido.crs)

gdf_localidad_sub = gdf_localidad[['LocNombre', 'geometry']].rename(columns={'LocNombre': 'localidad'})
gdf_enriquecido = gpd.sjoin(gdf_enriquecido, gdf_localidad_sub, how='left', predicate='intersects')
if 'index_right' in gdf_enriquecido.columns:
    gdf_enriquecido = gdf_enriquecido.drop(columns=['index_right'])
print(f"3. Registros tras sjoin con localidad: {len(gdf_enriquecido)}")

gdf_enriquecido = gdf_enriquecido.drop_duplicates(subset=['id_original'], keep='first')
gdf_enriquecido = gdf_enriquecido.drop(columns=['id_original'])
print(f"4. Registros tras limpiar duplicados por ID original: {len(gdf_enriquecido)}")

gdf_enriquecido = gdf_enriquecido.dropna(subset=['barrio', 'localidad', 'v_ref', 'shape_area', 'shape_len'])
print(f"5. Registros finales tras eliminar nulos esenciales: {len(gdf_enriquecido)}")

pd.DataFrame(gdf_enriquecido).to_csv('valor_referencia_historico_tabular.csv', index=False)
gdf_enriquecido.to_file('valor_referencia_historico_espacial.gpkg', driver='GPKG')
print("¡Proceso completado y archivos exportados con éxito!")

1. Registros en gdf_maestro inicial: 477077
2. Registros tras sjoin con barrio: 516447
3. Registros tras sjoin con localidad: 522201
4. Registros tras limpiar duplicados por ID original: 477077
5. Registros finales tras eliminar nulos esenciales: 477029
¡Proceso completado y archivos exportados con éxito!


In [ ]:
df = pd.read_csv('valor_referencia_historico_tabular.csv')
print("Localidades:", sorted(df['localidad'].dropna().unique()))
print(f"Total de localidades: {df['localidad'].nunique()}")
print(f"Total de barrios únicos: {df['barrio'].nunique()}")

/tmp/ipykernel_1510/3330608249.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('valor_referencia_historico_tabular.csv')


Localidades: ['ANTONIO NARIÑO', 'BARRIOS UNIDOS', 'BOSA', 'CANDELARIA', 'CHAPINERO', 'CIUDAD BOLIVAR', 'ENGATIVA', 'FONTIBON', 'KENNEDY', 'LOS MARTIRES', 'PUENTE ARANDA', 'RAFAEL URIBE URIBE', 'SAN CRISTOBAL', 'SANTA FE', 'SUBA', 'SUMAPAZ', 'TEUSAQUILLO', 'TUNJUELITO', 'USAQUEN', 'USME']
Total de localidades: 20
Total de barrios únicos: 1039


In [ ]:
df.head(10)

,mancodigo,v_ref,shape_area,shape_len,ano,geometry,barrio,localidad
0,003201039,2200000.0,7.318068e-07,0.004886,2025,POLYGON ((-74.08379815899991 4.591266770000061...,SAN BERNARDO,SANTA FE
1,002432009,3150000.0,2.083963e-07,0.002359,2025,POLYGON ((-74.15059074699991 4.573440389000041...,ARBORIZADORA BAJA,CIUDAD BOLIVAR
2,005657092,720000.0,7.810987e-08,0.001136,2025,"POLYGON ((-74.1463936799999 4.711376722000068,...",SABANA DEL DORADO,ENGATIVA
3,006402005,2360000.0,2.819338e-08,0.000717,2025,POLYGON ((-74.13848724699994 4.684827686000062...,LA CABANA FONTIBON,FONTIBON
4,002521015,650000.0,8.719964e-08,0.001492,2025,POLYGON ((-74.14006067299994 4.559116009000093...,LUCERO ALTO,CIUDAD BOLIVAR
5,005637079,2400000.0,3.180436e-06,0.009100,2025,"POLYGON ((-74.1380452009999 4.707592588000068,...",AEROPUERTO EL DORADO,FONTIBON
6,006402060,2360000.0,3.336503e-07,0.002704,2025,POLYGON ((-74.13881981499992 4.684557087000087...,LA CABANA FONTIBON,FONTIBON
7,002521014,650000.0,5.799995e-08,0.001092,2025,POLYGON ((-74.14034279599991 4.558895254000049...,LUCERO ALTO,CIUDAD BOLIVAR
8,002414027,1040000.0,3.073642e-07,0.002589,2025,POLYGON ((-74.16948457199993 4.582224375000067...,ISMAEL PERDOMO,CIUDAD BOLIVAR
9,002515011,2300000.0,3.419954e-07,0.002615,2025,POLYGON ((-74.14776032299994 4.563349932000051...,SAN FRANCISCO,CIUDAD BOLIVAR


# Transmilenio

In [1]:
import os
import zipfile
import glob
import re
import pandas as pd
import geopandas as gpd
from osgeo import gdal

# 1. Descomprimir el archivo zip de Transmilenio si es necesario
os.makedirs('temp_transmilenio', exist_ok=True)
for f in glob.glob('*transmilenio*.zip'):  # Ajusta el nombre si tu archivo tiene otro prefijo
    with zipfile.ZipFile(f, 'r') as z:
        z.extractall('temp_transmilenio')

# 2. Localizar y leer el archivo .shp de las estaciones
archivo_transmilenio = glob.glob('temp_transmilenio/**/*.shp', recursive=True)[0]
gdf_transmilenio = gpd.read_file(archivo_transmilenio)

print("CRS de Transmilenio:", gdf_transmilenio.crs)
print("Número de estaciones encontradas:", len(gdf_transmilenio))
print("Columnas disponibles:", gdf_transmilenio.columns.tolist())

# Visualizar las primeras filas
display(gdf_transmilenio.head(5))

CRS de Transmilenio: EPSG:3116
Número de estaciones encontradas: 153
Columnas disponibles: ['observ', 'longitud', 'latitud', 'objectid', 'num_est', 'nom_est', 'id_trazado', 'long_est', 'ancho_est', 'area_est', 'esta_oper', 'eta_oper', 'tipo_esta', 'ub_est', 'num_acc', 'acc_esp', 'acc_depr', 'acc_puent', 'cod_nodo', 'cap_biart', 'num_vag', 'cap_art', 'globalid', 'created_us', 'created_da', 'last_edite', 'last_edi_1', 'geometry']


,observ,longitud,latitud,objectid,num_est,nom_est,id_trazado,long_est,ancho_est,area_est,...,cod_nodo,cap_biart,num_vag,cap_art,globalid,created_us,created_da,last_edite,last_edi_1,geometry
0,Valores de capacidad de buses en proceso de ac...,-74.074575,4.666353,624,07103,AV. Chile,TZ008,173.64,4.0,694.56,...,7103,120,3,224,{88CDEE8D-FC67-43E6-B91B-1385BB3BD488},None,None,None,None,POINT (1000325.465 1007757.54)
1,Valores de capacidad de buses en proceso de ac...,-74.099046,4.646876,625,06103,CAN,TZ016,96.48,3.0,289.44,...,6103,120,2,140,{D7AC3687-AAC2-4153-801B-6DB705329D88},None,None,None,None,POINT (997610.17 1005603.861)
2,Valores de capacidad de buses en proceso de ac...,-74.078710,4.645398,626,07106,Campín - UAN,TZ008,122.82,4.0,491.28,...,7106,120,2,152,{992C8526-1970-4933-9EB9-826D966F6999},None,None,None,None,POINT (999866.611 1005440.294)
3,Valores de capacidad de buses en proceso de ac...,-74.071500,4.610899,627,10008,San Diego,TZ018,348.09,4.0,1392.36,...,10008,144,2,144,{667C3BFD-2FBE-40E7-AC18-8D327C4F17DF},None,None,None,None,POINT (1000666.64 1001625.403)
4,Valores de capacidad de buses en proceso de ac...,-74.096018,4.699151,628,04100,Granja - Kr 77,TZ005,385.00,4.0,1540.00,...,4100,144,4,248,{5BB6ECCC-FCBC-474C-8399-84E490AC3502},None,None,None,None,POINT (997946.377 1011384.46)


# Librerias

In [ ]:
import os
import zipfile
import glob
import re
import numpy as np
import pandas as pd
import geopandas as gpd
from osgeo import gdal

gdal.SetConfigOption("SHAPE_RESTORE_SHX", "YES")


# 0. Extracción de datos

In [2]:
# ==============================================================================
# PASO 0: EXTRACCIÓN AUTOMÁTICA DE ARCHIVOS COMPRIMIDOS
# ==============================================================================
os.makedirs('data_historica', exist_ok=True)
for f in glob.glob('*.zip'):
    if 'sector' not in f.lower() and 'transmilenio' not in f.lower():
        with zipfile.ZipFile(f, 'r') as z:
            z.extractall(os.path.join('data_historica', os.path.splitext(f)[0]))

os.makedirs('temp_sector', exist_ok=True)
for f in glob.glob('*sector*.zip'):
    with zipfile.ZipFile(f, 'r') as z:
        z.extractall('temp_sector')

os.makedirs('temp_transmilenio', exist_ok=True)
for f in glob.glob('*transmilenio*.zip'):
    with zipfile.ZipFile(f, 'r') as z:
        z.extractall('temp_transmilenio')

print("--- EXTRACCIÓN COMPLETADA ---")

--- EXTRACCIÓN COMPLETADA ---


# 1. Unificación Localidad - Barrio - Geometria

In [7]:
# ==============================================================================
# PASO 1: DICCIONARIO ESPACIAL BASE (LOCALIDAD - BARRIO - GEOMETRÍA)
# ==============================================================================
archivo_sector = glob.glob('temp_sector/**/*.shp', recursive=True)[0]
gdf_sector = gpd.read_file(archivo_sector)

gdf_localidad = gpd.read_file('loca.gpkg')
gdf_localidad = gdf_localidad.to_crs(gdf_sector.crs)

gdf_sector = gdf_sector.rename(columns={'SCANOMBRE': 'barrio'})

gdf_barrios_base = gpd.sjoin(gdf_sector[['barrio', 'geometry']], gdf_localidad[['LocNombre', 'geometry']], how='left', predicate='intersects')
gdf_barrios_base = gdf_barrios_base.rename(columns={'LocNombre': 'localidad'})
gdf_barrios_base = gdf_barrios_base.drop(columns=['index_right'])

# Deduplicar polígonos de barrio
gdf_barrios_base = gdf_barrios_base.drop_duplicates(subset=['barrio']).reset_index(drop=True)

print(f"\n[PASO 1] Tamaño del Diccionario Espacial Base (Barrios): {gdf_barrios_base.shape[0]} filas, {gdf_barrios_base.shape[1]} columnas")


[PASO 1] Tamaño del Diccionario Espacial Base (Barrios): 1150 filas, 3 columnas


In [8]:
gdf_barrios_base.head(5)

,barrio,geometry,localidad
0,EL BOSQUE,"POLYGON ((-74.10015 4.50545, -74.10016 4.50525...",USME
1,BILBAO,"POLYGON ((-74.11315 4.75187, -74.11371 4.751, ...",SUBA
2,CIUDAD KENNEDY CENTRAL,"POLYGON ((-74.15601 4.61652, -74.15601 4.61657...",KENNEDY
3,CONCEPCION NORTE,"POLYGON ((-74.06568 4.65636, -74.06511 4.65727...",CHAPINERO
4,COLON,"POLYGON ((-74.11964 4.61964, -74.11899 4.62043...",PUENTE ARANDA


# 2. Consolidación valores historicos de metro cuadrado

In [9]:
# ==============================================================================
# PASO 2: CONSOLIDACIÓN DE VALORES DE REFERENCIA HISTÓRICOS (VHI)
# ==============================================================================
lista_gdfs = []
for shp in glob.glob('data_historica/**/*.shp', recursive=True):
    gdf = gpd.read_file(shp)
    match = re.search(r'20\d{2}', shp)
    ano = int(match.group(0)) if match else None
    cols = {c.lower(): c for c in gdf.columns}
    datos = {
        'mancodigo': gdf[cols.get('mancodigo')] if cols.get('mancodigo') else None,
        'v_ref': gdf[cols.get('v_ref') or cols.get('valor_refe') or cols.get('valref')] if (cols.get('v_ref') or cols.get('valor_refe') or cols.get('valref')) else None,
        'ano': ano,
        'geometry': gdf['geometry']
    }
    lista_gdfs.append(gpd.GeoDataFrame(datos, crs=gdf.crs))

gdf_maestro = pd.concat(lista_gdfs, ignore_index=True)
print(f"[PASO 2a] Tamaño de Registros de Manzanas Históricas (gdf_maestro): {gdf_maestro.shape[0]} filas, {gdf_maestro.shape[1]} columnas")

gdf_barrios_base = gdf_barrios_base.to_crs(gdf_maestro.crs)

# Cruce espacial de manzanas con diccionario de barrios
gdf_historico = gpd.sjoin(gdf_maestro, gdf_barrios_base[['barrio', 'localidad', 'geometry']], how='inner', predicate='intersects')
if 'index_right' in gdf_historico.columns:
    gdf_historico = gdf_historico.drop(columns=['index_right'])

# Mediana del valor de referencia por barrio y año
df_vhi_barrio = gdf_historico.groupby(['localidad', 'barrio', 'ano'])['v_ref'].median().reset_index()
print(f"[PASO 2b] Tamaño del Consolidado VHI por Barrio y Año: {df_vhi_barrio.shape[0]} filas, {df_vhi_barrio.shape[1]} columnas")

[PASO 2a] Tamaño de Registros de Manzanas Históricas (gdf_maestro): 518935 filas, 4 columnas
[PASO 2b] Tamaño del Consolidado VHI por Barrio y Año: 13147 filas, 4 columnas


In [10]:
df_vhi_barrio.head(5)

,localidad,barrio,ano,v_ref
0,ANTONIO NARIÑO,CARACAS,2015,1300000.0
1,ANTONIO NARIÑO,CARACAS,2016,1400000.0
2,ANTONIO NARIÑO,CARACAS,2017,1700000.0
3,ANTONIO NARIÑO,CARACAS,2018,1700000.0
4,ANTONIO NARIÑO,CARACAS,2019,1750000.0


# 3. Extracción datos transmilenio

In [11]:
# ==============================================================================
# PASO 3: DENSIDAD DE INFRAESTRUCTURA DE TRANSMILENIO
# ==============================================================================
archivo_transmilenio = glob.glob('temp_transmilenio/**/*.shp', recursive=True)[0]
gdf_transmilenio = gpd.read_file(archivo_transmilenio)
print(f"[PASO 3a] Estaciones de Transmilenio leídas: {gdf_transmilenio.shape[0]} filas, {gdf_transmilenio.shape[1]} columnas")

if gdf_barrios_base.crs != gdf_transmilenio.crs:
    gdf_transmilenio = gdf_transmilenio.to_crs(gdf_barrios_base.crs)

# Conteo espacial de estaciones por barrio
estaciones_por_barrio = gpd.sjoin(gdf_transmilenio, gdf_barrios_base[['barrio', 'geometry']], how='inner', predicate='intersects')
conteo_estaciones = estaciones_por_barrio.groupby('barrio').size().reset_index(name='num_estaciones_transmilenio')
print(f"[PASO 3b] Barrios con al menos una estación de Transmilenio: {conteo_estaciones.shape[0]} barrios")

[PASO 3a] Estaciones de Transmilenio leídas: 153 filas, 28 columnas
[PASO 3b] Barrios con al menos una estación de Transmilenio: 111 barrios


In [17]:
conteo_estaciones

,barrio,num_estaciones_transmilenio
0,ABRAHAM LINCOLN,1
1,ACEVEDO TEJADA,2
2,ALTOS DE CHOZICA,1
3,AUTOPISTA MUZU,1
4,AUTOPISTA MUZU ORIENTAL,1
...,...,...
106,SIMON BOLIVAR,1
107,SOCIEGO,1
108,SOSIEGO SUR,1
109,SURAMERICA,1


# 4. Cálculo CAGR 2015-2026

In [13]:
# ==============================================================================
# PASO 4: CÁLCULO DE LA TASA DE CRECIMIENTO ANUAL COMPUESTA (CAGR 2015-2026)
# ==============================================================================
df_cagr_base = df_vhi_barrio[df_vhi_barrio['ano'].isin([2015, 2026])]
df_pivot = df_cagr_base.pivot(index=['localidad', 'barrio'], columns='ano', values='v_ref').reset_index()
df_pivot = df_pivot.rename(columns={2015: 'v_ref_2015', 2026: 'v_ref_2026'})

# Calcular CAGR de 11 años (2015 a 2026)
n_anos = 2026 - 2015
df_pivot['cagr_15_26'] = (df_pivot['v_ref_2026'] / df_pivot['v_ref_2015']) ** (1 / n_anos) - 1
print(f"[PASO 4] Tamaño de la tabla de CAGR (2015-2026): {df_pivot.shape[0]} filas, {df_pivot.shape[1]} columnas")

[PASO 4] Tamaño de la tabla de CAGR (2015-2026): 1103 filas, 5 columnas


In [14]:
df_pivot

ano,localidad,barrio,v_ref_2015,v_ref_2026,cagr_15_26
0,ANTONIO NARIÑO,CARACAS,1300000.00,3200000.0,0.085336
1,ANTONIO NARIÑO,EDUARDO SANTOS,900000.00,3000000.0,0.115667
2,ANTONIO NARIÑO,EL VERGEL,912500.00,3000000.0,0.114268
3,ANTONIO NARIÑO,LA FRAGUITA,1000000.00,3000000.0,0.105032
4,ANTONIO NARIÑO,SANTANDER,1000000.00,3150000.0,0.109944
...,...,...,...,...,...
1098,USME,VILLA ISRAEL,200000.00,295200.0,0.036028
1099,USME,VILLABEL,135000.00,393034.0,0.102023
1100,USME,YOMASA,230000.00,480000.0,0.069170
1101,USME,YOMASA NORTE,34001.50,170124.0,0.157630


# 5. Unificación datos

In [18]:
# ==============================================================================
# PASO 5: UNIFICACIÓN TOTAL, ORDENAMIENTO Y EXPORTACIÓN
# ==============================================================================
matriz_final = gdf_barrios_base.copy()

# Fusión con CAGR y valores de referencia
matriz_final = matriz_final.merge(df_pivot[['barrio', 'v_ref_2015', 'v_ref_2026', 'cagr_15_26']], on='barrio', how='left')

# Fusión con Transmilenio
matriz_final = matriz_final.merge(conteo_estaciones, on='barrio', how='left')
matriz_final['num_estaciones_transmilenio'] = matriz_final['num_estaciones_transmilenio'].fillna(0).astype(int)

# Manejo de nulos en CAGR si un barrio no tiene serie completa
matriz_final['cagr_15_26'] = matriz_final['cagr_15_26'].fillna(0)

# ---> ORDENAR LAS COLUMNAS SEGÚN LO SOLICITADO <---
columnas_ordenadas = [
    'localidad',
    'barrio',
    'v_ref_2015',
    'v_ref_2026',
    'cagr_15_26',
    'num_estaciones_transmilenio',
    'geometry'
]
matriz_final = matriz_final[columnas_ordenadas]

# Asegurar que se mantenga como GeoDataFrame espacial
matriz_final = gpd.GeoDataFrame(matriz_final, geometry='geometry')

# Exportación de archivos finales ordenados
matriz_final.to_csv('matriz_barrios_unificada.csv', index=False)
matriz_final.to_file('matriz_barrios_unificada.gpkg', driver='GPKG')

print("\n¡Matriz unificada, ordenada y exportada con éxito!")
print(f"Columnas finales actuales: {matriz_final.columns.tolist()}")


¡Matriz unificada, ordenada y exportada con éxito!
Columnas finales actuales: ['localidad', 'barrio', 'v_ref_2015', 'v_ref_2026', 'cagr_15_26', 'num_estaciones_transmilenio', 'geometry']


In [19]:
matriz_final

,localidad,barrio,v_ref_2015,v_ref_2026,cagr_15_26,num_estaciones_transmilenio,geometry
0,USME,EL BOSQUE,290000.0,660000.0,0.077625,0,"POLYGON ((-74.10015 4.50545, -74.10016 4.50525..."
1,SUBA,BILBAO,750000.0,2050000.0,0.095719,0,"POLYGON ((-74.11315 4.75187, -74.11371 4.751, ..."
2,KENNEDY,CIUDAD KENNEDY CENTRAL,1000000.0,2640000.0,0.092264,0,"POLYGON ((-74.15601 4.61652, -74.15601 4.61657..."
3,CHAPINERO,CONCEPCION NORTE,1400000.0,4380000.0,0.109255,1,"POLYGON ((-74.06568 4.65636, -74.06511 4.65727..."
4,PUENTE ARANDA,COLON,1300000.0,3170000.0,0.084407,0,"POLYGON ((-74.11964 4.61964, -74.11899 4.62043..."
...,...,...,...,...,...,...,...
1145,CIUDAD BOLIVAR,MEISSEN,64000.0,1395000.0,0.323342,0,"POLYGON ((-74.13463 4.56098, -74.13427 4.56086..."
1146,RAFAEL URIBE URIBE,LOS MOLINOS,450000.0,2600000.0,0.172873,0,"POLYGON ((-74.10753 4.55803, -74.10742 4.55802..."
1147,SUBA,EL PLAN,1400000.0,3798483.0,0.094983,0,"POLYGON ((-74.06395 4.74181, -74.06391 4.74181..."
1148,CIUDAD BOLIVAR,ISMAEL PERDOMO,400000.0,1050000.0,0.091698,0,"POLYGON ((-74.16717 4.58155, -74.16718 4.58155..."


In [21]:
# ==============================================================================
# TOP DE BARRIOS CON MÁS ESTACIONES DE TRANSMILENIO (DESDE MATRIZ_FINAL)
# ==============================================================================

# 1. Filtrar los barrios que tienen más de 0 estaciones y ordenar de forma descendente
top_barrios_transmilenio = matriz_final[matriz_final['num_estaciones_transmilenio'] > 0] \
    .sort_values(by='num_estaciones_transmilenio', ascending=False) \
    .reset_index(drop=True)

# 2. Seleccionar las columnas clave para el reporte
columnas_reporte = ['localidad', 'barrio', 'num_estaciones_transmilenio', 'v_ref_2026', 'cagr_15_26']

print("="*85)
print("TOP DE BARRIOS CON MAYOR DENSIDAD DE ESTACIONES DE TRANSMILENIO EN BOGOTÁ")
print("="*85)

# Mostrar el Top 15 (puedes cambiar el número si deseas ver más o menos)
print(top_barrios_transmilenio[columnas_reporte].head(15).to_string(index=False))
print("="*85)

# Opcional: Guardar este top en un archivo CSV independiente para tus anexos o reportes de tesis
top_barrios_transmilenio[columnas_reporte].to_csv('top_barrios_transmilenio.csv', index=False)
print("Reporte exportado exitosamente como 'top_barrios_transmilenio.csv'")

TOP DE BARRIOS CON MAYOR DENSIDAD DE ESTACIONES DE TRANSMILENIO EN BOGOTÁ
         localidad                      barrio  num_estaciones_transmilenio  v_ref_2026  cagr_15_26
     SAN CRISTOBAL                SAN BERNARDO                            3   2211000.0    0.125886
          FONTIBON CIUDAD SALITRE NOR-ORIENTAL                            3   4770000.0    0.125742
    CIUDAD BOLIVAR             PARQUE EL TUNAL                            3   1114805.0    0.087088
          SANTA FE                   SAN DIEGO                            2   4995000.0    0.109039
          ENGATIVA                  SANTA ROSA                            2   4003750.0    0.075383
       TEUSAQUILLO              ACEVEDO TEJADA                            2   4050000.0    0.108829
    CIUDAD BOLIVAR     CORREDOR FERREO DEL SUR                            2   3077000.0    0.196317
          SANTA FE                 LA FAVORITA                            2   3050000.0    0.106693
          ENGATIVA        